# S&P 500 Data Preparation & Exploratory Data Analysis (EDA)

In [ ]:
# Import the needed libraries
import numpy as np 
import pandas as pd 
import yfinance as yf
# Setting Plotly backend plotting for pandas 
pd.options.plotting.backend = 'plotly'
import plotly.io as pio
pio.templates.default = 'plotly_dark'
import holidays 


In [ ]:
# import raw data from yfinance
raw = yf.download('^GSPC', start='2000-01-01')

[*********************100%***********************]  1 of 1 completed


In [ ]:
# check our loaded data 
raw

Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Date,,,,,
2000-01-03,1455.219971,1478.000000,1438.359985,1469.250000,931800000
2000-01-04,1399.420044,1455.219971,1397.430054,1455.219971,1009000000
2000-01-05,1402.109985,1413.270020,1377.680054,1399.420044,1085500000
2000-01-06,1403.449951,1411.900024,1392.099976,1402.109985,1092300000
2000-01-07,1441.469971,1441.469971,1400.729980,1403.449951,1225200000
...,...,...,...,...,...
2026-04-28,7138.799805,7152.520020,7115.169922,7133.740234,4900650000
2026-04-29,7135.950195,7145.629883,7107.859863,7131.609863,5123100000


In [ ]:
# No duplicate index check
# yfinance occasionally returns duplicate dates
assert raw.index.duplicated().sum() == 0, "Duplicate dates found in index"

### 📊 Understanding OHLCV Market Data

Each row in the dataset represents a **single trading day** in the market.

---

**Open**  
The first recorded price at the start of the trading session (**9:30 AM EST**).  
It often reflects **overnight news, macroeconomic developments, and shifts in investor sentiment** while markets were closed.

---

**High & Low**  
The **maximum (High)** and **minimum (Low)** prices reached during the trading session.

- Together, they define the **intraday range**
- This provides insight into **market volatility and uncertainty**
- Larger ranges typically indicate **strong reactions to new information or increased trading activity**

---

**Close** ⭐ *(Most Important)*  
The final price recorded at the end of the trading session (**4:00 PM EST**).

- Represents the market’s **final consensus on valuation**
- The **primary variable used in financial analysis and modeling**
- Most time series models are built using:
  - Closing prices, or
  - Returns derived from closing prices

---

**Volume**  
Represents the level of trading activity during the day.

- For indices like the **S&P 500**, volume is **not directly observed**
- It is **derived or provided by the data source**
- As a result:
  - It may be **less reliable**
  - It is **not typically used as a primary feature in modeling**

## Data Inspection 

In [ ]:
# Check the DataSet shape
raw.shape

(6623, 5)

In [ ]:
type(raw.index)

pandas.DatetimeIndex

In [ ]:
# Check for missing values 
raw.isnull().sum()

Price   Ticker
Close   ^GSPC     0
High    ^GSPC     0
Low     ^GSPC     0
Open    ^GSPC     0
Volume  ^GSPC     0
dtype: int64

In [ ]:
# Check Data description 
summary = raw.describe()
summary

Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
count,6623.000000,6623.000000,6623.000000,6623.000000,6.623000e+03
mean,2327.217827,2340.064273,2312.613047,2326.846136,3.445466e+09
std,1533.781555,1540.500482,1525.785156,1533.413008,1.525549e+09
min,676.530029,695.270020,666.789978,679.280029,0.000000e+00
25%,1213.494995,1220.210022,1205.409973,1213.075012,2.340060e+09
50%,1552.010010,1556.390015,1543.839966,1551.689941,3.543790e+09
75%,2932.260010,2944.114990,2917.909912,2930.709961,4.290650e+09
max,7230.120117,7272.520020,7229.319824,7234.540039,1.145623e+10


The minimum value of the **Volume** column is **0**, which is unexpected for a confirmed trading day.

 This warrants further investigation to assess whether it represents a data integrity issue or a legitimate market condition.

In [ ]:
# Inspect your current column structure
raw.columns

MultiIndex([( 'Close', '^GSPC'),
            (  'High', '^GSPC'),
            (   'Low', '^GSPC'),
            (  'Open', '^GSPC'),
            ('Volume', '^GSPC')],
           names=['Price', 'Ticker'])

In [ ]:
# Flatten multi-level columns

if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)
raw = raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

# Validate the dataset flatten
print("Columns after flattening:", raw.columns.tolist())
raw.head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2000-01-03,1455.219971,1478.000000,1438.359985,1469.250000,931800000
2000-01-04,1399.420044,1455.219971,1397.430054,1455.219971,1009000000
2000-01-05,1402.109985,1413.270020,1377.680054,1399.420044,1085500000
2000-01-06,1403.449951,1411.900024,1392.099976,1402.109985,1092300000
2000-01-07,1441.469971,1441.469971,1400.729980,1403.449951,1225200000


### 🧩 Column Structure Adjustment: Flattening Multi-Level Headers

The dataset initially contains **multi-level column headers** due to the structure returned by the data source (*yfinance*), where each column is paired with the ticker symbol.

While this format is useful when working with **multiple assets**, it introduces unnecessary complexity in a **single-index context**.

---

**Why this matters**

- Complicates column selection and referencing  
- Increases the risk of errors in:
  - Feature engineering  
  - Data transformations  
  - Model input preparation  

---

**Action Taken**

- The multi-level columns were **flattened to a single level**
- This ensures:
  - Simpler column access  
  - Cleaner data manipulation  
  - Greater consistency across the pipeline  

---

**Key Insight**

> Standardizing the dataset structure early improves **robustness, readability, and maintainability** of the data pipeline.

In [ ]:
# How many days have a volume of 0
raw[raw['Volume'] == 0]

Price,Close,High,Low,Open,Volume
Date,,,,,
2023-05-24,4115.240234,4132.959961,4103.97998,4132.959961,0


In [ ]:
fig = raw['Volume'].plot(title='S&P 500 Daily Volume')
fig.update_layout(
    showlegend=False,
    yaxis_title='Volume',
    xaxis_title='Date'
)
fig.show()

In [ ]:
# programmatically verified zero-volume days against US holiday calendars
us_holidays = holidays.US()

zero_vol_dates = raw[raw['Volume'] == 0].index
for date in zero_vol_dates:
    holiday = us_holidays.get(date)
    print(f"{date.date()} | Holiday: {holiday if holiday else 'None — anomalous'}")

2023-05-24 | Holiday: None — anomalous


### ⚠️ Data Integrity Check: Zero-Volume Observation

During data validation, a **zero-volume entry** was identified on:

- **Date:** May 24, 2024

---

**Verification**

- Cross-checked against US market holidays using the `holidays` library  
- Confirmed: **Not a market holiday** → valid trading day  

---

**Assessment**

- The **S&P 500 is an index**, not a directly traded asset  
- Volume is:
  - **Derived / synthetic**
  - Dependent on the data provider (*yfinance*)  
- A zero value on a valid trading day indicates a **data integrity issue**, not a real market condition  

---

**Action Taken**

- The observation was **removed from the dataset**
- Rationale:
  - Prevent contamination of downstream analysis  
  - Avoid bias in:
    - Return calculations  
    - Volatility estimates  
    - Model training  

---

**Key Insight**

> External financial data sources must be **validated**, not blindly trusted.

In [ ]:
# We drop the rows that has Zero volume by 
raw = raw[raw['Volume']  > 0]

In [ ]:
# Confirm no zero-volume rows remain
raw[raw['Volume'] == 0]

Price,Close,High,Low,Open,Volume
Date,,,,,


The latest observation may represent **partial intraday data**, depending on when the API request was made.

Including such incomplete data can introduce bias into:
- **Return calculations**
- **Volatility estimates**

To ensure analytical consistency, this observation is excluded from the dataset.

In [ ]:

raw = raw.iloc[:-1]

In [ ]:
# Confirm the last row
raw.tail()

Price,Close,High,Low,Open,Volume
Date,,,,,
2026-04-27,7173.910156,7178.740234,7146.720215,7152.720215,4783750000
2026-04-28,7138.799805,7152.520020,7115.169922,7133.740234,4900650000
2026-04-29,7135.950195,7145.629883,7107.859863,7131.609863,5123100000
2026-04-30,7209.009766,7219.830078,7126.149902,7161.750000,5723790000
2026-05-01,7230.120117,7272.520020,7229.319824,7234.540039,4847390000


To assess whether the time series is suitable for **financial time series modeling**, 

price data is transformed into **log returns**, as returns exhibit more stable statistical properties than raw prices.

The resulting series is then analyzed to evaluate its behavior and suitability for modeling.

In [ ]:
# We create a new column and calculate the daily log Returns

raw['log_returns'] = np.log(raw['Close']/ raw['Close'].shift(1))



In [ ]:
# Double check our results

raw[['Close', 'log_returns']].head(10)

Price,Close,log_returns
Date,,
2000-01-03,1455.219971,NaN
2000-01-04,1399.420044,-0.039099
2000-01-05,1402.109985,0.001920
2000-01-06,1403.449951,0.000955
2000-01-07,1441.469971,0.026730
2000-01-10,1457.599976,0.011128
2000-01-11,1438.560059,-0.013149
2000-01-12,1432.250000,-0.004396
2000-01-13,1449.680054,0.012096


We used Log daily returns to stablize the variance through our data, which is a core assumption for stationarity assumption
for applying our statsitical models(ARIMA, SARIMA, PROPHET)
And the log return has a property called time additive, the additivity makes log returns much easier to work mathmetically, 
which is the standard in quantitive finance

In [ ]:
# I drop the first row as we can't compute daily returns for the first day of our data 
raw = raw.dropna(subset=['log_returns'])

# To confirm that all NaN values dropped:
raw['log_returns'].isna().sum()

np.int64(0)

In [ ]:
raw['log_returns'].describe()

count    6620.000000
mean        0.000242
std         0.012179
min        -0.127652
25%        -0.004734
50%         0.000639
75%         0.005878
max         0.109572
Name: log_returns, dtype: float64

The mean of the log_returns is almost zero, which satisfy the stationarity part of the data
Std (Standard Deviation):  this is average daily volatility, about 1.2% per day


In [ ]:
# We check the min and max dates for the daily returns :
# Find the actual dates of extreme moves
raw['log_returns'].nsmallest(5)

Date
2020-03-16   -0.127652
2020-03-12   -0.099945
2008-10-15   -0.094695
2008-12-01   -0.093537
2008-09-29   -0.092190
Name: log_returns, dtype: float64

In [ ]:
raw['log_returns'].nlargest(5)

Date
2008-10-13    0.109572
2008-10-28    0.102457
2025-04-09    0.090895
2020-03-24    0.089683
2020-03-13    0.088808
Name: log_returns, dtype: float64

These moves exceed typical daily returns by multiple standard deviation

The largest single-day gains:

2008 dominates the top gains — October 13 and 28, 2008 were both massive bailout-driven relief rallies during the financial crisis
April 9, 2025 — that's your Trump tariff pause
March 2020 — two entries, both COVID-era "dead cat bounces" as markets briefly rebounded during the crash

The largest single-day losses:

March 16, 2020 — the worst day in entire dataset at -12.7%, the Monday after the first COVID lockdown weekend
March 12, 2020 — two days earlier, WHO officially declared a pandemic
The rest are all 2008 — September through December, the heart of the financial crisis

In [ ]:
# Visualize log returns over time
fig = raw['log_returns'].plot(title='S&P 500 Daily Log Returns (2000–Present)')
fig.update_layout(
    showlegend=False,
    yaxis_title='Log Return',
    xaxis_title='Date'
)
fig.show()


The Chart Confirms Volatility Clustering, Major spikes in returns on both ways on 
Financial Crisis 2008, during Covid crash on 2020 and slightly smaller spike on early 2026 
While period between 2013 and 2019 is lower volatility and stable time.

The model I build will detect the spikes in the magnitude not the reasons.

In [ ]:
# Visualize Close Price over time
fig = raw['Close'].plot(title='S&P 500 Close Price (2000–Present)')
fig.update_layout(
    showlegend=False,
    yaxis_title='S&P 500',
    xaxis_title='Date'
)
fig.show()